# 03 — Backtest Tearsheet

Quintile long–short backtest on lagged characteristics (e.g. Amihud ``illiq_signal``).

Weights: long sleeve $+0.5$, short sleeve $-0.5$ (dollar neutral, unit gross).

In [ ]:
import polars as pl
import matplotlib.pyplot as plt

from src.data.store import read_partitioned_parquet
from src.data.paths import PROCESSED_RETURNS_DIR, PROCESSED_FACTORS_DIR
from src.backtest import BacktestConfig, run_backtest, run_quantile_spread

chars = read_partitioned_parquet(PROCESSED_FACTORS_DIR / "characteristics")
print("characteristic rows:", chars.height)
chars.head()

In [ ]:
signal = "illiq_signal" if chars.height and "illiq_signal" in chars.columns else None
if signal:
    cfg = BacktestConfig(
        signal_col=signal,
        kind="quintile",
        long_high=True,
        weighting="equal",
        frequency="month_end",
    )
    result = run_backtest(chars, cfg)
    print(result.metrics)
    display(result.returns.tail())
else:
    result = None
    print("No characteristics panel — run Phase 2–3 first.")

In [ ]:
if result is not None and result.returns.height:
    pdf = result.returns.to_pandas()
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(pdf["trade_date"], pdf["wealth"], label="L/S wealth")
    ax.set_title("Cumulative wealth (net)")
    ax.legend()
    plt.show()

    to = result.turnover.to_pandas()
    fig, ax = plt.subplots(figsize=(9, 3))
    ax.bar(to["trade_date"].astype(str), to["turnover_two_way"])
    ax.set_title("Two-way turnover by rebalance")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

In [ ]:
if signal and chars.height:
    spread = run_quantile_spread(chars, signal, kind="quintile")
    cum = (
        spread.sort(["quantile", "trade_date"])
        .with_columns((1 + pl.col("q_ret")).cum_prod().over("quantile").alias("q_wealth"))
    )
    for q in range(1, 6):
        sub = cum.filter(pl.col("quantile") == q).to_pandas()
        plt.plot(sub["trade_date"], sub["q_wealth"], label=f"Q{q}")
    plt.legend()
    plt.title("Quintile cumulative wealth")
    plt.show()